# 05 — Reducción Dimensional y Análisis de Variables Latentes

## 1. Marco Teórico
Cuando se estudian múltiples variables meteorológicas y contaminantes estrechamente correlacionados, el análisis individual puede resultar redundante y confuso. Empleamos tres métodos complementarios de reducción de dimensiones:

### Análisis de Componentes Principales (PCA)
PCA es un método geométrico no paramétrico que proyecta las variables observadas en direcciones ortogonales (componentes principales) de forma que maximizan de manera decreciente la variabilidad total explicada. Las componentes son combinaciones lineales de las variables originales y se derivan de la descomposición espectral de la matriz de correlación muestral $\mathbf{R}$:
$$\mathbf{R} = \mathbf{V} \boldsymbol{\Lambda} \mathbf{V}^T$$
donde las columnas de $\mathbf{V}$ son las cargas (*loadings*) y la diagonal de $\boldsymbol{\Lambda}$ contiene los autovalores $\lambda_i$ (varianza de cada componente).

### Probabilistic PCA (PPCA)
PPCA formula PCA bajo un modelo probabilístico latente con variables latentes gaussianas $\mathbf{z} \sim N(\mathbf{0}, \mathbf{I})$ y un término de ruido isotrópico $\mathbf{e} \sim N(\mathbf{0}, \sigma^2 \mathbf{I})$:
$$\mathbf{x} = \mathbf{W}\mathbf{z} + \boldsymbol{\mu} + \mathbf{e}$$
Esto permite modelar formalmente la incertidumbre instrumental de los datos y estimar la varianza del ruido del sensor de calidad del aire a través de $\sigma^2$.

### Análisis Factorial (FA) con Rotación Varimax
A diferencia de PCA que solo busca maximizar varianza, el Análisis Factorial asume que existe una estructura subyacente de pocos factores comunes latentes que causan las correlaciones observadas. Las variables observadas se modelan como:
$$\mathbf{x} = \mathbf{L}\mathbf{f} + \mathbf{u}$$
donde $\mathbf{L}$ son las cargas de los factores comunes, $\mathbf{f}$ son los factores, y $\mathbf{u}$ representa la variabilidad específica o ruido único de cada variable ($uniqueness_j = \Psi_{jj}$).

Para lograr que las componentes sean físicamente interpretables, aplicamos una **Rotación Ortogonal Varimax** (implementada a mano en nuestro pipeline). La rotación Varimax maximiza la suma de las varianzas de las cargas al cuadrado dentro de cada factor:
$$V = \frac{1}{p} \sum_{j=1}^k \left[ p \sum_{i=1}^p L_{ij}^4 - \left( \sum_{i=1}^p L_{ij}^2 \right)^2 \right]$$
Esto rota los ejes del factor de manera que cada variable cargue muy fuertemente en un solo factor latente y casi cero en los demás, facilitando su identificación.

*Nota metodológica:* La variable discreta/bimodal 'Lluvia' ha sido retirada de todos estos análisis para mantener la coherencia teórica del supuesto de normalidad y continuidad multivariable latente.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
os.chdir(ROOT)

import pandas as pd
import IPython.display as display
import importlib.util
spec = importlib.util.spec_from_file_location('dimred', ROOT / 'Scripts/11_dimensionality_reduction.py')
dimred = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dimred)

df, x = dimred.load_data()
print(f"Dimensiones de la matriz de variables ambientales Z (sin lluvia): {x.shape[0]} filas x {x.shape[1]} columnas")
print(f"Variables continuas incluidas: {dimred.ENV_VARS}")

## 2. Ajuste de Modelos y Criterio de Selección de Componentes
Ajustamos PCA para observar los autovalores y la varianza explicada acumulada. Esto nos permite aplicar criterios de selección de número de factores:
1. **Criterio de Kaiser:** Retener componentes con autovalor $\lambda_i > 1$.
2. **Criterio del Scree Plot (Codo):** Identificar el punto de inflexión donde la varianza explicada deja de caer drásticamente.
3. **Criterio del 80% de variabilidad explicada.**

In [ ]:
pca, pca_scores, pca_loadings = dimred.fit_pca(x)
ppca, ppca_scores = dimred.fit_ppca(x)
fa, fa_scores, fa_loadings, fa_rotated_loadings, fa_rotated_scores = dimred.fit_factor_analysis(x)

explained, pca_loadings_df = dimred.save_pca_outputs(pca, pca_loadings)
fa_loadings_df, uniqueness = dimred.save_factor_outputs(fa, fa_rotated_loadings)
scores = dimred.save_scores(df, pca_scores, fa_rotated_scores)
comparison = dimred.save_comparison(pca, ppca, fa, explained)

print("=== VARIANZA EXPLICADA PCA ===")
display.display(explained.round(4))

# Graficar Scree Plot
dimred.plot_scree(explained)
from IPython.display import Image
Image(filename='Figures/Dimensionality_Reduction/01_pca_scree_plot.png')

## 3. Visualización del Espacio Proyectado: PCA Biplot 2D y Dispersión 3D
Graficamos las dos primeras componentes principales del PCA en un Biplot 2D tradicional.

Además, al habernos quedado con exactamente 3 componentes principales (que explican el 80.63% de la varianza total), podemos realizar una **representación tridimensional (3D)** de todas las observaciones, proyectando los registros en el espacio $[PC1, PC2, PC3]$ y coloreando los puntos según pertenezcan al grupo de Baja Diversidad (Q1) o Alta Diversidad (Q3). Esto permite inspeccionar visualmente la separación física de los micro-hábitats de las aves urbanas de Bogotá.

In [ ]:
# Graficar Biplot 2D y heatmaps de cargas
dimred.plot_pca_biplot(scores, pca_loadings_df)
dimred.plot_loadings_heatmap(pca_loadings_df, '03_pca_loadings_heatmap.png', 'Cargas PCA')
dimred.plot_loadings_heatmap(fa_loadings_df, '04_factor_analysis_varimax_loadings.png', 'Cargas Factor Analysis Varimax')

from IPython.display import display, Image
print("=== PROYECCIÓN BIDIMENSIONAL (BIPLOT PCA) ===")
display(Image(filename='Figures/Dimensionality_Reduction/02_pca_biplot_q1_q3.png'))

# Graficar interactivo 3D con Plotly Express (rotativo)
import plotly.express as px
import numpy as np

subset = scores[scores[dimred.GROUP_COL].isin(['Baja_Q1', 'Alta_Q3'])].copy()
subset = subset.sample(min(len(subset), 2200), random_state=42)
subset['Diversidad'] = subset[dimred.GROUP_COL].map({'Baja_Q1': 'Baja Diversidad (Q1)', 'Alta_Q3': 'Alta Diversidad (Q3)'})

fig = px.scatter_3d(
    subset,
    x='PC1',
    y='PC2',
    z='PC3',
    color='Diversidad',
    color_discrete_map={'Baja Diversidad (Q1)': '#c84630', 'Alta Diversidad (Q3)': '#2a9d8f'},
    title='Proyección Tridimensional PCA (Micro-hábitats de Aves) — INTERACTIVO (Arrastra para rotar)',
    opacity=0.6,
    height=600
)
fig.update_layout(
    scene=dict(
        xaxis_title='PC1',
        yaxis_title='PC2',
        zaxis_title='PC3'
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)
fig.show()

print("\n=== CALIDAD DE REPRESENTACIÓN (HEATMAP DE CARGAS PCA) ===")
display(Image(filename='Figures/Dimensionality_Reduction/03_pca_loadings_heatmap.png'))

## Interpretación de Componentes Principales (PCA)
En el Análisis de Componentes Principales, las componentes se ordenan estrictamente de manera decreciente según la varianza explicada. No hay rotación, por lo que las cargas representan la proyección ortogonal de mayor varianza:
* **PC1 (Factor Climático):** Explica el **45.87%** de la varianza total. Carga fuertemente con Humedad (0.89), Temperatura (-0.86), Ozono (-0.85) y Radiación Solar (-0.81). Captura la macro-variabilidad meteorológica de la ciudad.
* **PC2 (Factor de Material Particulado):** Explica el **27.52%** de la varianza. Carga fuertemente con $PM_{10}$ (0.88) y $PM_{2.5}$ (0.84). Representa la polución por partículas sólidas en suspensión.
* **PC3 (Factor de Gases de Combustión):** Explica el **7.24%** de la varianza. Carga moderadamente con $NO_2$ (0.43) y $CO$ (0.39), capturando emisiones gaseosas locales.

## 4. PPCA y Modelo Generativo de Variables Latentes
A diferencia del enfoque geométrico de PCA, el Análisis de Componentes Principales Probabilístico (PPCA) asume un **modelo generativo**. Postula que las observaciones de las variables ambientales son generadas por 3 *causas* o *variables latentes* subyacentes inobservables, más un ruido aleatorio de medición ($\sigma^2 \approx 0.291$).

Matemáticamente, la estimación por máxima verosimilitud asume:
$$\mathbf{x} = \mathbf{W} \mathbf{z} + \boldsymbol{\mu} + \boldsymbol{\epsilon}$$
Donde $\mathbf{z} \sim \mathcal{N}(0, \mathbf{I})$ son las variables latentes (clima, material particulado, gases), $\mathbf{W}$ es la matriz de cargas que relaciona factores con variables, y $\boldsymbol{\epsilon} \sim \mathcal{N}(0, \sigma^2 \mathbf{I})$ es el ruido isotrópico. La estimación de $\sigma^2$ equivale al promedio de los autovalores no retenidos.

Analizando cómo estas variables latentes influyen en los sensores:
1. **Variable Latente 1 (El Clima):** Genera la co-ocurrencia observada entre niveles de humedad y radiación/temperatura. Cuando el valor latente de este 'clima' sube, esperamos probabilísticamente que la estación reporte una caída en la temperatura y un aumento en la humedad.
2. **Variable Latente 2 (La Fuente de Partículas):** Actúa como el emisor subyacente de polución física. Cuando esta variable generativa está activa, aumenta la probabilidad de que los sensores de la RMCAB detecten simultáneamente altos niveles de $PM_{2.5}$ y $PM_{10}$.
3. **Variable Latente 3 (La Fuente de Gases):** Es el proceso generativo de las emisiones de combustión. Al incrementarse, fuerza matemáticamente la subida conjunta en las lecturas de los gases $NO_2$ y $CO$.

A continuación calculamos y mostramos la matriz de pesos generativos $\mathbf{W}$, que nos indica la fuerza (y dirección) con la que cada Factor Latente subyacente moldea el valor observado de las variables ambientales originales.

In [ ]:
import numpy as np
import pandas as pd
import IPython.display as display

# Matriz de pesos generativos W en PPCA: W = U_q (L_q - sigma^2 I)^{1/2}
sigma2 = ppca.noise_variance_
eigenvalues = ppca.explained_variance_
W_ppca = ppca.components_.T * np.sqrt(np.maximum(eigenvalues - sigma2, 0))

ppca_weights_df = pd.DataFrame(
    W_ppca,
    index=dimred.CONTINUOUS_VARS,
    columns=['Latente 1 (Clima)', 'Latente 2 (Partículas)', 'Latente 3 (Gases)']
)
print(f"PPCA Noise Variance Estimate (sigma2): {sigma2:.4f}\n")
print("=== MATRIZ DE PESOS GENERATIVOS W (PPCA) ===")
display.display(ppca_weights_df.round(4))

## Interpretación de los Ejes Factoriales y Conclusiones Ecológicas
El PPCA nos permite afirmar, desde una óptica probabilística, que la compleja red de correlaciones ambientales que detectamos se puede simplificar en 3 verdaderos motores físicos (Clima, Partículas y Gases), y que la varianza restante es atribuible al error o ruido de los propios instrumentos de medición. Las aves habitan en este espacio tridimensional probabilístico donde cada dimensión actúa como una fuerza ecológica independiente.

## 5. Cargas Rotadas del Análisis Factorial (Varimax) y Análisis de Varianza Comunal y Unicidad
Revisamos las cargas del Análisis Factorial tras la rotación Varimax con 3 factores comunes latentes.

En el Análisis Factorial, la varianza total de cada variable estandarizada se descompone en dos componentes ortogonales:
1. **Comunalidad (Varianza Comunal o $h_j^2$):** Es la proporción de la varianza de la variable $j$ que es explicada por los factores comunes latentes. Se calcula como la suma de los cuadrados de las cargas factoriales de esa variable en todos los factores retenidos.
2. **Unicidad (Uniqueness o $\psi_j$):** Es la proporción de la varianza que **no** es explicada por los factores comunes ($\psi_j = 1 - h_j^2$). Representa la varianza específica de esa variable particular más el error de medición aleatorio.

Examinar la comunalidad y unicidad nos permite evaluar la calidad de representación: variables con alta comunalidad (baja unicidad) están muy bien explicadas por el modelo dimensional latente.

In [ ]:
print("=== CARGAS FACTOR ANALYSIS (VARIMAX ROTATED, 3 FACTORES) ===")\ndisplay.display(fa_loadings_df.round(3))

## 6. Comparación de Métodos de Reducción Dimensional
Mostramos la tabla de comparación del ajuste global de los tres algoritmos (PCA, PPCA y Factor Analysis), evaluando la asunción latente de cada uno y la varianza total explicada.

In [ ]:
print("=== COMPARACIÓN DE MÉTODOS DE REDUCCIÓN DE DIMENSIONES ===")\ndisplay.display(comparison)

## Interpretación de los Ejes Factoriales y Conclusiones Ecológicas

A diferencia de PCA o PPCA, en el **Análisis Factorial con Rotación Varimax**, la varianza compartida se redistribuye para maximizar la separación de las cargas (acercándolas a 0 o 1). Esto reordena los factores dando prioridad a la interpretabilidad física más limpia, separando por completo las señales que el PCA mezclaba:

* **Factor 1 (Contaminación por Material Particulado):** Carga fuertemente con $PM_{2.5}$ (0.97) y $PM_{10}$ (0.80). Este factor aísla la señal de polvo y polución física de manera pura.
* **Factor 2 (Clima / Estabilidad Térmica):** Carga fuertemente con Temperatura (0.93), Humedad Relativa (-0.91), Radiación Solar (0.86), Ozono (0.79) y Viento (0.56). Aísla las condiciones meteorológicas del día.
* **Factor 3 (Gases de Combustión Local):** Carga fuertemente con $NO_2$ (0.71) y $CO$ (0.65). Separa de manera independiente los gases tóxicos, provenientes típicamente de escapes de vehículos.

### Conclusiones Metodológicas
- **Comunalidades Aceptables:** Al retener 3 factores latentes, el *uniqueness* promedio es de alrededor de 0.272 (lo que implica una comunalidad promedio del 72.8%). Las variables conservan comunalidades altas ($>0.65$), a excepción del Viento, lo cual valida la eficacia del modelo para representar el sistema original.
- **Ventaja del Análisis Factorial (FA) sobre PCA/PPCA:** Mientras que el PCA nos sirvió para proyectar los datos y visualizar la separación general de las observaciones (útil geométricamente), y el PPCA nos dio una visión del proceso generativo y el ruido instrumental, el **Análisis Factorial Rotado** nos entrega la interpretación de variables más clara. Nos demuestra estadísticamente que los micro-hábitats urbanos de Bogotá (y la exposición de sus aves) están gobernados por tres dimensiones totalmente independientes: el nivel de partículas, el clima del sector, y la concentración de gases de combustión.